In [0]:
spark


In [0]:
# Project: Databricks NYC Taxi Lakehouse Analytics
# Notebook: 01_data_ingestion_bronze
# Objective: Read raw files from Unity Catalog Volume

from pyspark.sql import functions as F

taxi_file_path = "/Volumes/workspace/default/nyc_taxi_lakehouse/yellow_tripdata_2024-01.parquet"
zones_file_path = "/Volumes/workspace/default/nyc_taxi_lakehouse/taxi_zone_lookup.csv"

print("Paths configured successfully.")

Paths configured successfully.


In [0]:
# Read raw files

df_taxi_raw = spark.read.parquet(taxi_file_path)

df_zones_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(zones_file_path)
)

print("Taxi rows:", df_taxi_raw.count())
print("Taxi columns:", len(df_taxi_raw.columns))

print("Zones rows:", df_zones_raw.count())
print("Zones columns:", len(df_zones_raw.columns))

Taxi rows: 2964624
Taxi columns: 19
Zones rows: 265
Zones columns: 4


In [0]:
# Preview raw taxi data

display(df_taxi_raw.limit(10))

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
2,2024-01-01T00:57:55.000,2024-01-01T01:17:43.000,1,1.72,1,N,186,79,2,17.7,1.0,0.5,0.0,0.0,1.0,22.7,2.5,0.0
1,2024-01-01T00:03:00.000,2024-01-01T00:09:36.000,1,1.8,1,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
1,2024-01-01T00:17:06.000,2024-01-01T00:35:01.000,1,4.7,1,N,236,79,1,23.3,3.5,0.5,3.0,0.0,1.0,31.3,2.5,0.0
1,2024-01-01T00:36:38.000,2024-01-01T00:44:56.000,1,1.4,1,N,79,211,1,10.0,3.5,0.5,2.0,0.0,1.0,17.0,2.5,0.0
1,2024-01-01T00:46:51.000,2024-01-01T00:52:57.000,1,0.8,1,N,211,148,1,7.9,3.5,0.5,3.2,0.0,1.0,16.1,2.5,0.0
1,2024-01-01T00:54:08.000,2024-01-01T01:26:31.000,1,4.7,1,N,148,141,1,29.6,3.5,0.5,6.9,0.0,1.0,41.5,2.5,0.0
2,2024-01-01T00:49:44.000,2024-01-01T01:15:47.000,2,10.82,1,N,138,181,1,45.7,6.0,0.5,10.0,0.0,1.0,64.95,0.0,1.75
1,2024-01-01T00:30:40.000,2024-01-01T00:58:40.000,0,3.0,1,N,246,231,2,25.4,3.5,0.5,0.0,0.0,1.0,30.4,2.5,0.0
2,2024-01-01T00:26:01.000,2024-01-01T00:54:12.000,1,5.44,1,N,161,261,2,31.0,1.0,0.5,0.0,0.0,1.0,36.0,2.5,0.0
2,2024-01-01T00:28:08.000,2024-01-01T00:29:16.000,1,0.04,1,N,113,113,2,3.0,1.0,0.5,0.0,0.0,1.0,8.0,2.5,0.0


In [0]:
# Preview taxi zone lookup data

display(df_zones_raw.limit(10))

LocationID,Borough,Zone,service_zone
1,EWR,Newark Airport,EWR
2,Queens,Jamaica Bay,Boro Zone
3,Bronx,Allerton/Pelham Gardens,Boro Zone
4,Manhattan,Alphabet City,Yellow Zone
5,Staten Island,Arden Heights,Boro Zone
6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone
7,Queens,Astoria,Boro Zone
8,Queens,Astoria Park,Boro Zone
9,Queens,Auburndale,Boro Zone
10,Queens,Baisley Park,Boro Zone


In [0]:
# Check taxi raw schema

df_taxi_raw.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [0]:
from pyspark.sql import functions as F

taxi_file_path = "/Volumes/workspace/default/nyc_taxi_lakehouse/yellow_tripdata_2024-01.parquet"
zones_file_path = "/Volumes/workspace/default/nyc_taxi_lakehouse/taxi_zone_lookup.csv"

df_taxi_raw = spark.read.parquet(taxi_file_path)

df_zones_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(zones_file_path)
)

print("DataFrames loaded successfully.")

DataFrames loaded successfully.


In [0]:
# Create Bronze Delta tables

bronze_taxi_table = "workspace.default.bronze_yellow_taxi_trips"
bronze_zones_table = "workspace.default.bronze_taxi_zones"

df_taxi_raw.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(bronze_taxi_table)

df_zones_raw.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(bronze_zones_table)

print("Bronze tables created successfully.")

Bronze tables created successfully.


In [0]:
# Validate Bronze Delta tables

display(spark.sql("SHOW TABLES IN workspace.default"))

database,tableName,isTemporary
default,bronze_customers,false
default,bronze_ecommerce_sales,false
default,bronze_incremental_sales,false
default,bronze_orders,false
default,bronze_payments,false
default,bronze_products,false
default,bronze_taxi_zones,false
default,bronze_yellow_taxi_trips,false
default,gold_category_revenue,false
default,gold_customer_last_order,false


In [0]:
# Validate only project Bronze tables

display(
    spark.sql("""
        SHOW TABLES IN workspace.default
    """)
    .filter("tableName LIKE 'bronze_%taxi%'")
)

database,tableName,isTemporary
default,bronze_taxi_zones,false
default,bronze_yellow_taxi_trips,false


In [0]:
# Count rows in Bronze Delta tables

bronze_taxi_count = spark.table("workspace.default.bronze_yellow_taxi_trips").count()
bronze_zones_count = spark.table("workspace.default.bronze_taxi_zones").count()

print("Bronze taxi rows:", bronze_taxi_count)
print("Bronze zones rows:", bronze_zones_count)

Bronze taxi rows: 2964624
Bronze zones rows: 265
